In [29]:
from typesafe_sdk import AsyncTypeSafeClient, Choice, Noul, Score

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

import pandas as pd

from pathlib import Path

from PIL import Image, ImageDraw
from typesafe_sdk import Box, Image as TSImage, TypeSafeClient

In [30]:
import json
import logging

SOURCE = Path("cars2.jpg")          # your input image
OUTPUT = Path("cars-boxed.jpg")    # final image: kept boxes only
DEBUG_OUTPUT = Path("cars-debug.jpg")  # every tile candidate, including rejected ones
LOG_FILE = Path("run_log.json")    # per-tile record of the whole run
TILE_DIR = Path("tiles")           # where the tile crops are written
TARGET = "red car"                 # describe the object to box
GRID = 3                           # split the image into GRID x GRID tiles
OVERLAP = 0.15                     # extra overlap between neighbouring tiles, as a fraction of the image size
PAD = 0.0                          # optional padding, as a fraction of the image size
MIN_CONFIDENCE = 0.3               # ignore boxes the model is unsure about
NOUL_THRESHOLD = 0.3               # presence score (0..1) needed to keep a tile's box
IOU_THRESHOLD = 0.5                # boxes overlapping more than this are the same object

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
log = logging.getLogger("jev-boxes")
log.info("Config: target=%r grid=%d overlap=%.2f pad=%.2f min_conf=%.2f noul_threshold=%.2f iou=%.2f",
         TARGET, GRID, OVERLAP, PAD, MIN_CONFIDENCE, NOUL_THRESHOLD, IOU_THRESHOLD)

image = Image.open(SOURCE).convert("RGB")
width, height = image.size
log.info("Image %s is %dx%d px", SOURCE, width, height)

# Box returns one box per question and always returns a box, so cover the image
# with overlapping tiles and ask a Box plus a presence Noul on each tile.
TILE_DIR.mkdir(exist_ok=True)
tiles = {}
step = 1 / GRID
for row in range(GRID):
    for col in range(GRID):
        left = max(0.0, col * step - OVERLAP / 2)
        top = max(0.0, row * step - OVERLAP / 2)
        right = min(1.0, (col + 1) * step + OVERLAP / 2)
        bottom = min(1.0, (row + 1) * step + OVERLAP / 2)
        name = f"tile_{row}_{col}"
        path = TILE_DIR / f"{name}.jpg"
        crop_px = (
            round(left * width),
            round(top * height),
            round(right * width),
            round(bottom * height),
        )
        image.crop(crop_px).save(path, "JPEG", quality=92)
        tiles[name] = (left, top, right, bottom, path)
        log.info("%s bounds frac=(%.3f, %.3f, %.3f, %.3f) px=%s -> %s",
                 name, left, top, right, bottom, crop_px, path)

# One request per tile: a single-image state keeps each Noul and Box question
# pointed at the image it was written for
answers = {}
with TypeSafeClient() as client:
    for name, tile in tiles.items():
        result = client.system_one(
            model="jev-preview",       # Box questions need Jev 1.14
            state={"tile": TSImage(tile[4])},
            questions={
                "present": Noul(instructions=f"Is there a {TARGET} in `tile`?"),
                "box": Box(instructions=f"the {TARGET} in `tile`"),
            },
            extra_headers={"TypeSafe-Danger-Zone-Images": "true"},
        )
        answers[name] = result.answers
        log.debug("%s raw answers: %r", name, result.answers)

log.info("Got answers for %d tiles (model=%s)", len(answers), result.model)

2026-10-15 12:02:48,725 INFO Config: target='red car' grid=3 overlap=0.15 pad=0.00 min_conf=0.30 noul_threshold=0.30 iou=0.50
2026-10-15 12:02:48,756 INFO Image cars2.jpg is 1300x957 px
2026-10-15 12:02:48,764 INFO tile_0_0 bounds frac=(0.000, 0.000, 0.408, 0.408) px=(0, 0, 531, 391) -> tiles\tile_0_0.jpg
2026-10-15 12:02:48,770 INFO tile_0_1 bounds frac=(0.258, 0.000, 0.742, 0.408) px=(336, 0, 964, 391) -> tiles\tile_0_1.jpg
2026-10-15 12:02:48,776 INFO tile_0_2 bounds frac=(0.592, 0.000, 1.000, 0.408) px=(769, 0, 1300, 391) -> tiles\tile_0_2.jpg
2026-10-15 12:02:48,784 INFO tile_1_0 bounds frac=(0.000, 0.258, 0.408, 0.742) px=(0, 247, 531, 710) -> tiles\tile_1_0.jpg
2026-10-15 12:02:48,794 INFO tile_1_1 bounds frac=(0.258, 0.258, 0.742, 0.742) px=(336, 247, 964, 710) -> tiles\tile_1_1.jpg
2026-10-15 12:02:48,803 INFO tile_1_2 bounds frac=(0.592, 0.258, 1.000, 0.742) px=(769, 247, 1300, 710) -> tiles\tile_1_2.jpg
2026-10-15 12:02:48,809 INFO tile_2_0 bounds frac=(0.000, 0.592, 0.408, 

In [31]:
# Whole-image count check: none, one, or several red cars
# (uses TARGET and SOURCE from the config cell above)
with TypeSafeClient() as client:
    count_result = client.system_one(
        model="jev-preview",
        state={"photo": TSImage(SOURCE)},
        questions={
            "count": Choice(
                instructions=f"Are there none, one, or several {TARGET}s in `photo`?",
                criteria={
                    "none": f"No {TARGET}s are visible in `photo`",
                    "one": f"Exactly one {TARGET} is visible in `photo`",
                    "several": f"Two or more {TARGET}s are visible in `photo`",
                },
            ),
        },
        extra_headers={"TypeSafe-Danger-Zone-Images": "true"},
    )

count = count_result.answers["count"]
log.info("Count question: %s (confidence %.2f)", count.choice, count.confidence)
log.info("Count probabilities: %s", {option: round(p, 2) for option, p in count.probabilities.items()})

2026-10-15 12:02:52,852 INFO HTTP Request: POST https://api.typesafe.ai/v1/systemone "HTTP/1.1 200 OK"
2026-10-15 12:02:52,854 INFO POST https://api.typesafe.ai/v1/systemone <- 200 in 386ms (request req_01a13ecccc9074539f34d4175afcf927)
2026-10-15 12:02:52,856 INFO Count question: several (confidence 0.62)
2026-10-15 12:02:52,858 INFO Count probabilities: {'none': 0.0, 'one': 0.25, 'several': 0.75}


In [32]:
def tile_to_image(edges, tile):
    # Convert edges from fractions of a tile to fractions of the full image
    left, top, right, bottom, _ = tile
    tile_w, tile_h = right - left, bottom - top
    return (
        left + edges.left * tile_w,
        top + edges.top * tile_h,
        left + edges.right * tile_w,
        top + edges.bottom * tile_h,
    )


def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union else 0.0


# One record per tile, updated with the decision made for it
decisions = []
for name, tile in tiles.items():
    presence = answers[name]["present"]
    box_answer = answers[name]["box"]
    box_image = tile_to_image(box_answer.box, tile)
    record = {
        "tile": name,
        "noul": presence.noul,
        "confidence": box_answer.confidence,
        "box_in_tile": [box_answer.box.left, box_answer.box.top, box_answer.box.right, box_answer.box.bottom],
        "box_in_image": [round(v, 4) for v in box_image],
        "raw": {"present": repr(presence), "box": repr(box_answer)},
    }
    if presence.noul is None:
        # Keep the record (with the raw answer) and fail after the log is written
        record["status"] = "error: Noul value is None"
    elif presence.noul < NOUL_THRESHOLD:
        record["status"] = "rejected: presence below threshold"
    elif box_answer.confidence < MIN_CONFIDENCE:
        record["status"] = "rejected: confidence below threshold"
    else:
        record["status"] = "candidate"
    log.info("%s: noul=%r confidence=%.2f box_in_image=%s -> %s",
             name, presence.noul, box_answer.confidence, record["box_in_image"], record["status"])
    decisions.append(record)

# Tiles overlap, so the same car can be found more than once: keep the most confident one
candidates = [r for r in decisions if r["status"] == "candidate"]
cars = []
for record in sorted(candidates, key=lambda r: r["confidence"], reverse=True):
    box = tuple(record["box_in_image"])
    duplicate_of = None
    for kept in cars:
        overlap = iou(box, tuple(kept["box_in_image"]))
        if overlap >= IOU_THRESHOLD:
            duplicate_of = (kept["tile"], overlap)
            break
    if duplicate_of:
        record["status"] = f"suppressed: IoU {duplicate_of[1]:.2f} with {duplicate_of[0]}"
        log.info("%s suppressed: IoU %.2f with %s", record["tile"], duplicate_of[1], duplicate_of[0])
    else:
        record["status"] = "kept"
        cars.append(record)

log.info("Final: %d %s(s) kept from %d candidates out of %d tiles",
         len(cars), TARGET, len(candidates), len(decisions))

LOG_FILE.write_text(json.dumps({
    "target": TARGET,
    "image": str(SOURCE),
    "size_px": [width, height],
    "config": {
        "grid": GRID, "overlap": OVERLAP, "pad": PAD, "min_confidence": MIN_CONFIDENCE,
        "noul_threshold": NOUL_THRESHOLD, "iou_threshold": IOU_THRESHOLD,
    },
    "tiles": decisions,
    "kept": [r["tile"] for r in cars],
}, indent=2), encoding="utf-8")
log.info("Wrote per-tile log to %s", LOG_FILE)

missing = [r["tile"] for r in decisions if r["status"].startswith("error")]
if missing:
    raise ValueError(f"Noul value is None for {missing}; see {LOG_FILE} for the raw answers")

2026-10-15 12:02:52,895 INFO tile_0_0: noul=0.11 confidence=0.06 box_in_image=[0.2654, 0.3185, 0.3879, 0.392] -> rejected: presence below threshold
2026-10-15 12:02:52,897 INFO tile_0_1: noul=0.15 confidence=0.05 box_in_image=[0.6643, 0.0531, 0.7223, 0.3961] -> rejected: presence below threshold
2026-10-15 12:02:52,899 INFO tile_0_2: noul=0.5 confidence=0.39 box_in_image=[0.8285, 0.1837, 0.9224, 0.3103] -> candidate
2026-10-15 12:02:52,902 INFO tile_1_0: noul=0.3 confidence=0.05 box_in_image=[0.1429, 0.3115, 0.2287, 0.7368] -> rejected: confidence below threshold
2026-10-15 12:02:52,905 INFO tile_1_1: noul=0.08 confidence=0.09 box_in_image=[0.4323, 0.5387, 0.645, 0.7368] -> rejected: presence below threshold
2026-10-15 12:02:52,907 INFO tile_1_2: noul=0.49 confidence=0.21 box_in_image=[0.8571, 0.2777, 0.9143, 0.3792] -> rejected: confidence below threshold
2026-10-15 12:02:52,908 INFO tile_2_0: noul=0.96 confidence=0.58 box_in_image=[0.1388, 0.7142, 0.2777, 0.9061] -> candidate
2026-10

In [33]:
# Final image: kept boxes only. Debug image: every tile candidate, kept in red,
# rejected or suppressed in yellow, labelled with tile name and confidence
debug_image = image.copy()
draw_final = ImageDraw.Draw(image)
draw_debug = ImageDraw.Draw(debug_image)
line_width = max(2, round(min(width, height) / 200))


def to_px(box):
    left, top, right, bottom = box
    left = max(0.0, left - PAD)
    top = max(0.0, top - PAD)
    right = min(1.0, right + PAD)
    bottom = min(1.0, bottom + PAD)
    return (
        round(left * width),
        round(top * height),
        round(right * width),
        round(bottom * height),
    )


for record in decisions:
    box_px = to_px(record["box_in_image"])
    kept = record["status"] == "kept"
    color = (255, 0, 0) if kept else (255, 215, 0)
    draw_debug.rectangle(box_px, outline=color, width=line_width if kept else max(1, line_width // 2))
    draw_debug.text((box_px[0] + 4, box_px[1] + 4), f"{record['tile']} {record['confidence']:.2f}", fill=color)
    if kept:
        draw_final.rectangle(box_px, outline=color, width=line_width)
    log.info("%s box_px=%s %s", record["tile"], box_px, record["status"])

image.save(OUTPUT, "JPEG", quality=92)
debug_image.save(DEBUG_OUTPUT, "JPEG", quality=92)
log.info("Saved %s (%d box(es)) and %s (all %d tiles)", OUTPUT, len(cars), DEBUG_OUTPUT, len(decisions))

2026-10-15 12:02:52,956 INFO tile_0_0 box_px=(345, 305, 504, 375) rejected: presence below threshold
2026-10-15 12:02:52,960 INFO tile_0_1 box_px=(864, 51, 939, 379) rejected: presence below threshold
2026-10-15 12:02:52,963 INFO tile_0_2 box_px=(1077, 176, 1199, 297) kept
2026-10-15 12:02:52,966 INFO tile_1_0 box_px=(186, 298, 297, 705) rejected: confidence below threshold
2026-10-15 12:02:52,968 INFO tile_1_1 box_px=(562, 516, 838, 705) rejected: presence below threshold
2026-10-15 12:02:52,970 INFO tile_1_2 box_px=(1114, 266, 1189, 363) rejected: confidence below threshold
2026-10-15 12:02:52,972 INFO tile_2_0 box_px=(180, 683, 361, 867) kept
2026-10-15 12:02:52,973 INFO tile_2_1 box_px=(342, 586, 392, 781) rejected: confidence below threshold
2026-10-15 12:02:52,975 INFO tile_2_2 box_px=(775, 570, 843, 824) kept
2026-10-15 12:02:53,011 INFO Saved cars-boxed.jpg (3 box(es)) and cars-debug.jpg (all 9 tiles)
